In [2]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [3]:
df = pd.read_csv('bookstore_apriori_500_rows.csv')

In [4]:
df.sample(5)

,Transaction_ID,Customer_ID,Book_ID,Book_Name,Category,Quantity,Order_Date
338,T068,C1068,11,Operating Systems,Systems,1,2026-03-09
435,T088,C1088,8,C++ Programming,Programming,1,2026-03-29
15,T004,C1004,1,Python Programming,Programming,1,2026-01-04
121,T025,C1025,2,Data Science Handbook,Data Science,1,2026-01-25
457,T092,C1092,12,Design and Analysis of Algorithms,Computer Science,1,2026-04-02


In [7]:
df['Transaction_ID'].nunique()

100

In [6]:
df['Book_Name'].nunique()

25

In [8]:
transactions = df.groupby('Transaction_ID')['Book_Name'].apply(lambda x : list(set(x)))

In [9]:
transactions.sample(5)

Transaction_ID
T012    [Database Management Systems, Machine Learning...
T054    [SQL Fundamentals, Artificial Intelligence, Op...
T069    [The Lord of the Rings, Clean Code, C++ Progra...
T076    [Design and Analysis of Algorithms, SQL Fundam...
T009    [Database Management Systems, Machine Learning...
Name: Book_Name, dtype: object

In [10]:
te = TransactionEncoder()

In [11]:
transaction_array = te.fit(transactions).transform(transactions)

In [12]:
basket = pd.DataFrame(data = transaction_array, columns=te.columns_)

In [13]:
basket.head()

,1984,Artificial Intelligence,Atomic Habits,C++ Programming,Clean Code,Computer Networks,Data Science Handbook,Database Management Systems,Deep Learning,Design and Analysis of Algorithms,Harry Potter,Ikigai,Java Programming,Machine Learning Basics,Operating Systems,Python Programming,Rich Dad Poor Dad,SQL Fundamentals,The Alchemist,The Great Gatsby,The Hobbit,The Lord of the Rings,The Psychology of Money,Think and Grow Rich,Web Development with HTML CSS JS
0,False,False,False,False,False,False,True,True,False,False,False,False,False,True,False,True,False,True,False,False,False,False,False,False,False
1,False,False,False,False,False,False,True,True,False,False,False,False,False,True,False,True,False,True,False,False,False,False,False,False,False
2,False,False,False,False,False,False,True,True,False,False,False,False,False,True,False,True,False,True,False,False,False,False,False,False,False
3,False,False,False,False,False,False,True,True,False,False,False,False,False,True,False,True,False,True,False,False,False,False,False,False,False
4,False,False,False,False,False,False,True,True,False,False,False,False,False,True,False,True,False,True,False,False,False,False,False,False,False


In [14]:
frequent_itemsets = apriori(df=basket, min_support=0.4, use_colnames=True)
display(frequent_itemsets)

,support,itemsets
0,0.74,frozenset({Data Science Handbook})
1,0.55,frozenset({Database Management Systems})
2,0.50,frozenset({Machine Learning Basics})
3,0.70,frozenset({Python Programming})
4,0.69,frozenset({SQL Fundamentals})
5,0.51,"frozenset({Database Management Systems, Data S..."
6,0.46,"frozenset({Machine Learning Basics, Data Scien..."
7,0.70,"frozenset({Python Programming, Data Science Ha..."
8,0.63,"frozenset({SQL Fundamentals, Data Science Hand..."
9,0.46,"frozenset({Machine Learning Basics, Database M..."


In [15]:
rules = association_rules(df= frequent_itemsets, metric='confidence', min_threshold=0.60)
display(rules)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({Database Management Systems}),frozenset({Data Science Handbook}),0.55,0.74,0.51,0.927273,1.253071,1.0,0.1030,3.575000,0.448802,0.653846,0.720280,0.808231
1,frozenset({Data Science Handbook}),frozenset({Database Management Systems}),0.74,0.55,0.51,0.689189,1.253071,1.0,0.1030,1.447826,0.776772,0.653846,0.309309,0.808231
2,frozenset({Machine Learning Basics}),frozenset({Data Science Handbook}),0.50,0.74,0.46,0.920000,1.243243,1.0,0.0900,3.250000,0.391304,0.589744,0.692308,0.770811
3,frozenset({Data Science Handbook}),frozenset({Machine Learning Basics}),0.74,0.50,0.46,0.621622,1.243243,1.0,0.0900,1.321429,0.752508,0.589744,0.243243,0.770811
4,frozenset({Python Programming}),frozenset({Data Science Handbook}),0.70,0.74,0.70,1.000000,1.351351,1.0,0.1820,inf,0.866667,0.945946,1.000000,0.972973
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,frozenset({Database Management Systems}),"frozenset({Machine Learning Basics, SQL Fundam...",0.55,0.46,0.45,0.818182,1.778656,1.0,0.1970,2.970000,0.972840,0.803571,0.663300,0.898221
176,frozenset({Machine Learning Basics}),"frozenset({Python Programming, SQL Fundamental...",0.50,0.51,0.45,0.900000,1.764706,1.0,0.1950,4.900000,0.866667,0.803571,0.795918,0.891176
177,frozenset({SQL Fundamentals}),"frozenset({Machine Learning Basics, Database M...",0.69,0.45,0.45,0.652174,1.449275,1.0,0.1395,1.581250,1.000000,0.652174,0.367589,0.826087
178,frozenset({Python Programming}),"frozenset({Machine Learning Basics, SQL Fundam...",0.70,0.45,0.45,0.642857,1.428571,1.0,0.1350,1.540000,1.000000,0.642857,0.350649,0.821429


In [16]:
result = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
result['antecedents'] = result['antecedents'].apply(', '.join)
result['consequents'] = result['consequents'].apply(', '.join)

In [17]:
display(result)

,antecedents,consequents,support,confidence,lift
0,Database Management Systems,Data Science Handbook,0.51,0.927273,1.253071
1,Data Science Handbook,Database Management Systems,0.51,0.689189,1.253071
2,Machine Learning Basics,Data Science Handbook,0.46,0.920000,1.243243
3,Data Science Handbook,Machine Learning Basics,0.46,0.621622,1.243243
4,Python Programming,Data Science Handbook,0.70,1.000000,1.351351
...,...,...,...,...,...
175,Database Management Systems,"Machine Learning Basics, SQL Fundamentals, Pyt...",0.45,0.818182,1.778656
176,Machine Learning Basics,"Python Programming, SQL Fundamentals, Database...",0.45,0.900000,1.764706
177,SQL Fundamentals,"Machine Learning Basics, Database Management S...",0.45,0.652174,1.449275
178,Python Programming,"Machine Learning Basics, SQL Fundamentals, Dat...",0.45,0.642857,1.428571


In [22]:
is_strong = (rules['lift']>=1.3) & (rules['support']>=0.8)
strong_rules = result[is_strong].copy()
weak_rules = result[~is_strong].copy()

In [23]:
strong_rules

,antecedents,consequents,support,confidence,lift


In [24]:
weak_rules

,antecedents,consequents,support,confidence,lift
0,Database Management Systems,Data Science Handbook,0.51,0.927273,1.253071
1,Data Science Handbook,Database Management Systems,0.51,0.689189,1.253071
2,Machine Learning Basics,Data Science Handbook,0.46,0.920000,1.243243
3,Data Science Handbook,Machine Learning Basics,0.46,0.621622,1.243243
4,Python Programming,Data Science Handbook,0.70,1.000000,1.351351
...,...,...,...,...,...
175,Database Management Systems,"Machine Learning Basics, SQL Fundamentals, Pyt...",0.45,0.818182,1.778656
176,Machine Learning Basics,"Python Programming, SQL Fundamentals, Database...",0.45,0.900000,1.764706
177,SQL Fundamentals,"Machine Learning Basics, Database Management S...",0.45,0.652174,1.449275
178,Python Programming,"Machine Learning Basics, SQL Fundamentals, Dat...",0.45,0.642857,1.428571
